# Python Notebook Example

Demonstrates reading from lakehouse with DuckDB/delta-rs and writing to warehouse via T-SQL.
Python notebooks start in ~5 seconds (no Spark cluster) and come with DuckDB, Polars, and delta-rs pre-installed.

In [ ]:
import sys
print(f'Python {sys.version}')

## Read lakehouse Delta table with delta-rs

In [ ]:
from deltalake import DeltaTable

# Option 1: Local path (requires attached lakehouse)
dt = DeltaTable('/lakehouse/default/Tables/my_table')
df = dt.to_pandas()
print(df.head())

# Option 2: ABFS path (any lakehouse; no attachment needed)
access_token = notebookutils.credentials.getToken('storage')
storage_options = {'bearer_token': access_token, 'use_fabric_endpoint': 'true'}

dt = DeltaTable(
    'abfss://<workspace-guid>@onelake.dfs.fabric.microsoft.com/<lakehouse-guid>/Tables/schema/table',
    storage_options=storage_options
)
df = dt.to_pandas()
print(df.head())

## Read lakehouse Delta table with DuckDB

In [ ]:
import duckdb
from deltalake import DeltaTable

# DuckDB can query delta-rs Arrow datasets with filter pushdown
access_token = notebookutils.credentials.getToken('storage')
storage_options = {'bearer_token': access_token, 'use_fabric_endpoint': 'true'}

dt = DeltaTable(
    'abfss://<workspace-guid>@onelake.dfs.fabric.microsoft.com/<lakehouse-guid>/Tables/schema/table',
    storage_options=storage_options
)
arrow_ds = dt.to_pyarrow_dataset()

result = duckdb.sql('SELECT count(*) as rows, max(date_key) as latest FROM arrow_ds').df()
print(result)

## Write to lakehouse with delta-rs

In [ ]:
from deltalake.writer import write_deltalake
import pandas as pd

df = pd.DataFrame({'id': [1, 2, 3], 'name': ['a', 'b', 'c']})
write_deltalake('/lakehouse/default/Tables/my_output_table', df, mode='overwrite')
print('Written to lakehouse')

## Query warehouse with T-SQL via notebookutils.data

In [ ]:
# connect_to_artifact supports: Warehouse (full DML), Lakehouse (read-only),
# SQLDatabase (full DML), MirroredDatabase (read-only)
with notebookutils.data.connect_to_artifact('WarehouseName') as conn:
    conn.query('CREATE TABLE dbo.test (id INT, name VARCHAR(100))')
    conn.query("INSERT INTO dbo.test VALUES (1, 'hello'), (2, 'world')")
    df = conn.query('SELECT * FROM dbo.test')
    print(df)